# Day 29 — Data Ingestion & Pipeline Initialization (Nigerian Retail Sales Analyzer)

This notebook covers Enterprise Data Ingestion, Multi-Region Synthetic Data Generation, Schema Enforcement, and Anomaly Injection

## Validation goals
- Initialize the Nigerian Retail Sales Analyzer project tracking multi-region commercial hubs (Balogun Market Lagos, Wuse Market Abuja, Kantin Kwari Kano, Computer Village Lagos, and Onitsha Main Market Anambra)
- Simulate a raw 5,000-row enterprise dataset featuring Naira pricing, quantities, dates, categories, and suppliers
- Inject intentional structural anomalies (missing prices, missing locations, and corrupt quantity flags like `-99`) to test cleaning pipelines
- Enforce explicit memory-efficient data types (`string`, `float32`, `int32`, `category`, `datetime64`) during ingestion via `pd.read_csv()`
- Export raw artifacts and validate memory optimization profiles

We begin the Nigerian Retail Sales Analyzer project by establishing a robust data ingestion pipeline. In real-world enterprise analytics, data rarely arrives pre-formatted; it is pulled from messy databases, multi-region APIs, or legacy CSV exports containing structural inconsistencies, missing fields, and unparsed timestamps.

For this project, we will simulate a multi-region retail dataset tracking transactions across major commercial hubs in Nigeria (e.g., Balogun Market Lagos, Wuse Market Abuja, Kantin Kwari Kano, Computer Village Lagos, and Onitsha Main Market, Anambra), featuring local product categories, pricing in Naira, and intentional anomalies to test our upcoming cleaning pipeline.

In [2]:
import numpy as np
import pandas as pd

# ==========================================
# Phase 1: Simulate Raw Multi-Region Retail Dataset
# ==========================================
np.random.seed(42)
n_rows = 5000

locations = [
    " Balogun Market, Lagos ", 
    "Wuse Market, Abuja", 
    "Kantin Kwari, Kano", 
    "Computer Village, Lagos", 
    "Onitsha Main Market, Anambra"
]

categories = ["Electronics", "Groceries", "Apparel", "Home Appliances"]

products_map = {
    "Electronics": ["Inverter 5kVA", "Solar Panel 300W", "Generator 2.5KVA", "Power Bank 20000mAh"],
    "Groceries": ["Bag of Rice 50kg", "Palm Oil 5L", "Garri 25kg", "Beans 10kg"],
    "Apparel": ["Ankara Fabric (6 yards)", "Senator Wear Material", "Leather Loafers"],
    "Home Appliances": ["Blender 3-in-1", "Standing Fan", "Gas Cooker 4-Burner"]
}

suppliers = ["Dangote Group", "Innoson Logistics", "Multichoice Hub", "Local Artisan Co", "Emzor Trade Ltd"]

random_locations = np.random.choice(locations, size=n_rows)
random_categories = np.random.choice(categories, size=n_rows)
random_products = [np.random.choice(products_map[cat]) for cat in random_categories]
random_suppliers = np.random.choice(suppliers, size=n_rows)

raw_prices = np.random.uniform(5000.0, 450000.0, size=n_rows)
raw_quantities = np.random.randint(1, 15, size=n_rows)
raw_dates = pd.date_range(start="2026-01-01", periods=n_rows, freq="h").astype(str)

df_raw = pd.DataFrame({
    "product": random_products,
    "price": raw_prices,
    "quantity": raw_quantities,
    "location": random_locations,
    "date": raw_dates,
    "supplier": random_suppliers,
    "category": random_categories
})

# Inject intentional anomalies for Day 30 cleaning practice
df_raw.loc[np.random.choice(n_rows, 50), "price"] = np.nan
df_raw.loc[np.random.choice(n_rows, 30), "location"] = np.nan
df_raw.loc[np.random.choice(n_rows, 20), "quantity"] = -99

# Save raw dataset directly into the data-analysis-project directory
df_raw.to_csv("raw_retail_sales.csv", index=False)
print(f"Successfully generated raw dataset with {len(df_raw)} rows.")


# ==========================================
# Phase 2: Optimized Data Ingestion
# ==========================================
schema_dtypes = {
    "product": "string",
    "price": "float32",
    "quantity": "int32",
    "location": "string",
    "supplier": "category",
    "category": "category"
}

df_ingested = pd.read_csv(
    "raw_retail_sales.csv",
    dtype=schema_dtypes,
    parse_dates=["date"]
)

print("\n--- Ingested DataFrame Info ---")
print(df_ingested.info())
print("\n--- Memory Usage ---")
print(df_ingested.memory_usage(deep=True))
print("\nFirst 3 Rows:")
print(df_ingested.head(3))

Successfully generated raw dataset with 5000 rows.

--- Ingested DataFrame Info ---
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   product   5000 non-null   string        
 1   price     4950 non-null   float32       
 2   quantity  5000 non-null   int32         
 3   location  4970 non-null   string        
 4   date      5000 non-null   datetime64[us]
 5   supplier  5000 non-null   category      
 6   category  5000 non-null   category      
dtypes: category(2), datetime64[us](1), float32(1), int32(1), string(2)
memory usage: 166.6 KB
None

--- Memory Usage ---
Index          132
product     322689
price        20000
quantity     20000
location    354925
date         40000
supplier      5493
category      5410
dtype: int64

First 3 Rows:
                   product          price  quantity  \
0               Beans 10kg  406758.031250        1